# Chapter 7: MCP サーバーを作る 【OpenAI版】

Chapter 5 で作ったツールを MCP (Model Context Protocol) サーバーとして公開し、クライアントから呼び出します。

※ MCP サーバーは LLM を使わないため、OpenAI/Bedrock の区別はありません。

## セットアップ

In [ ]:
%%bash
echo "Node.js: $(node -v)"
cd ../.. && npm install

## ポイント解説

### MCP サーバー（server.ts）

Chapter 5 のツールを `MCPServer` に渡して `startStdio()` するだけで、MCP サーバーが完成します。

```typescript
import { MCPServer } from "@mastra/mcp";
import { getCurrentDate, searchTopic } from "../chapter5/tools.js";

const server = new MCPServer({
  id: "blog-research-server",
  name: "Blog Research MCP Server",
  version: "1.0.0",
  tools: { getCurrentDate, searchTopic },  // ← Chapter 5 のツールをそのまま渡す
});

server.startStdio();  // ← stdio トランスポートで起動
```

⚠️ stdio トランスポートは標準入出力を MCP 通信に使うため、`console.log()` を使うとプロトコルが壊れます。

### MCP クライアント（run.ts）

`MCPClient` で server.ts を子プロセスとして起動し、ツールを呼び出します。

```typescript
import { MCPClient } from "@mastra/mcp";

const client = new MCPClient({
  servers: {
    blogResearch: {
      command: "npx",                        // ← server.ts を子プロセスで起動
      args: ["tsx", "src/chapter7/server.ts"],
    },
  },
});

const tools = await client.listTools();  // ← サーバーが公開しているツール一覧を取得
```

### MCP クライアント（Cursor 等）からの接続設定例

```json
{
  "mcpServers": {
    "blog-research": {
      "command": "npx",
      "args": ["tsx", "src/chapter7/server.ts"],
      "cwd": "/path/to/mastra-agent-tutorial"
    }
  }
}
```

## Chapter 7の実行

MCPClient が server.ts を子プロセスとして起動し、ツールを呼び出します。

In [ ]:
%%bash
cd ../.. && npm run ch7

## 観察ポイント

1. `listTools()` で取得したツール名が `blogResearch_getCurrentDate` のように **サーバー名_ツール名** になっている
2. Chapter 5 のツール（`getCurrentDate`, `searchTopic`）がそのまま MCP 経由で動いている
3. サーバー側のコードは `MCPServer` + `startStdio()` だけ — ツール定義の再利用がポイント

## コード（参考）

In [ ]:
%%bash
echo "=== server.ts (MCP サーバー) ==="
cat ../../src/chapter7/server.ts
echo ""
echo "=== run.ts (デモクライアント) ==="
cat ../../src/chapter7/run.ts

## おわり

お疲れさまでした！全チャプターが完了しました。